# Combined Task 2 + Task 3 Notes

This notebook is meant to be run and inspected. It shows how model depth, `N`, noise, image resolution, and regularization affect the results.

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

np.random.seed(42)
torch.manual_seed(42)


## Task 2 Helpers

In [ ]:
def make_task2_dataset(num_samples=1000, N=3, seed=42):
    rng = np.random.default_rng(seed)
    a = 0.3 + 0.5 * rng.random(num_samples)
    b = 0.3 + 0.5 * rng.random(num_samples)
    theta = np.pi * rng.random(num_samples)
    coeffs = np.zeros((num_samples, 2 * (N + 1)), dtype=np.float32)
    for i in range(num_samples):
        coeffs_complex = np.zeros(N + 1, dtype=np.complex128)
        coeffs_complex[0] = a[i] * b[i]
        for n in range(1, N + 1):
            coeffs_complex[n] = (a[i] - b[i]) * np.exp(1j * theta[i]) * (0.5 ** n)
        coeffs[i, :] = np.concatenate([np.real(coeffs_complex), np.imag(coeffs_complex)])
    targets = np.column_stack([a, b, theta]).astype(np.float32)
    return coeffs, targets

def split_train_val(X, Y, train_size=800):
    idx = np.arange(len(X))
    rng = np.random.default_rng(123)
    rng.shuffle(idx)
    return X[idx[:train_size]], X[idx[train_size:]], Y[idx[:train_size]], Y[idx[train_size:]]

def normalize_train_val(X_train, X_val):
    mu = X_train.mean(axis=0)
    sigma = X_train.std(axis=0) + 1e-8
    return (X_train - mu) / sigma, (X_val - mu) / sigma, mu, sigma

def build_regression_model(input_dim, output_dim, hidden_units, l2_strength=0.0):
    modules = []
    prev_dim = input_dim
    for units in hidden_units:
        modules.append(nn.Linear(prev_dim, units))
        modules.append(nn.ReLU())
        prev_dim = units
    modules.append(nn.Linear(prev_dim, output_dim))
    model = nn.Sequential(*modules)
    model.weight_decay = l2_strength
    return model

def count_params(model):
    return sum(param.numel() for param in model.parameters())

def train_and_score(X_train, X_val, Y_train, Y_val, hidden_units, epochs=8, l2_strength=0.0):
    X_train_n, X_val_n, _, _ = normalize_train_val(X_train, X_val)
    Y_train_n, Y_val_n, Y_mu, Y_sigma = normalize_train_val(Y_train, Y_val)
    model = build_regression_model(X_train.shape[1], Y_train.shape[1], hidden_units, l2_strength=l2_strength)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=l2_strength)
    criterion = nn.MSELoss()
    train_loader = DataLoader(
        TensorDataset(torch.tensor(X_train_n, dtype=torch.float32), torch.tensor(Y_train_n, dtype=torch.float32)),
        batch_size=32,
        shuffle=True,
    )
    val_tensor_X = torch.tensor(X_val_n, dtype=torch.float32)
    val_tensor_Y = torch.tensor(Y_val_n, dtype=torch.float32)
    val_losses = []
    for _ in range(epochs):
        model.train()
        for batch_X, batch_Y in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(batch_X), batch_Y)
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            val_losses.append(float(criterion(model(val_tensor_X), val_tensor_Y).item()))
    with torch.no_grad():
        pred_n = model(val_tensor_X).cpu().numpy()
    pred = pred_n * Y_sigma + Y_mu
    err = np.abs(pred - Y_val)
    return {
        'layers': str(hidden_units),
        'params': count_params(model),
        'final_val_loss': val_losses[-1],
        'mae_a': float(err[:, 0].mean()),
        'mae_b': float(err[:, 1].mean()),
        'mae_theta_deg': float(np.degrees(err[:, 2].mean())),
    }


## Task 2: Add or Remove Layers

We compare a shallow network, a middle network, and a deeper network.

In [ ]:
X, Y = make_task2_dataset(num_samples=1000, N=3, seed=42)
X_train, X_val, Y_train, Y_val = split_train_val(X, Y, train_size=800)
arch_results = [
    train_and_score(X_train, X_val, Y_train, Y_val, hidden_units=[32], epochs=8),
    train_and_score(X_train, X_val, Y_train, Y_val, hidden_units=[64, 32], epochs=8),
    train_and_score(X_train, X_val, Y_train, Y_val, hidden_units=[128, 64, 32], epochs=8),
]
pd.DataFrame(arch_results)

### Note

If the shallow model has the highest error, that is the underfitting case. If the deeper model only improves a little, the middle model is usually the better tradeoff.

## Task 2: Increase `N`

We compare the same architecture at two different coefficient orders.

In [ ]:
X3, Y3 = make_task2_dataset(num_samples=1000, N=3, seed=7)
X5, Y5 = make_task2_dataset(num_samples=1000, N=5, seed=7)
X3_train, X3_val, Y3_train, Y3_val = split_train_val(X3, Y3, train_size=800)
X5_train, X5_val, Y5_train, Y5_val = split_train_val(X5, Y5, train_size=800)
n_results = [
    {'N': 3, **train_and_score(X3_train, X3_val, Y3_train, Y3_val, hidden_units=[64, 32], epochs=8)},
    {'N': 5, **train_and_score(X5_train, X5_val, Y5_train, Y5_val, hidden_units=[64, 32], epochs=8)},
]
pd.DataFrame(n_results)

### Note

A larger `N` usually needs more capacity or more training. If the error rises, that is expected.

## Task 2: Add Noise

We train on clean inputs and test on noisy inputs.

In [ ]:
X, Y = make_task2_dataset(num_samples=1000, N=3, seed=99)
X_train, X_val, Y_train, Y_val = split_train_val(X, Y, train_size=800)
X_train_n, X_val_n, X_mu, X_sigma = normalize_train_val(X_train, X_val)
Y_train_n, Y_val_n, Y_mu, Y_sigma = normalize_train_val(Y_train, Y_val)
model = build_regression_model(X_train.shape[1], Y_train.shape[1], [64, 32])
model.fit(X_train_n, Y_train_n, epochs=8, batch_size=32, verbose=0, validation_data=(X_val_n, Y_val_n))
rng = np.random.default_rng(2024)
noise_rows = []
for noise in [0.0, 0.05, 0.10]:
    noisy_val = X_val + rng.normal(scale=noise, size=X_val.shape)
    noisy_val_n = (noisy_val - X_mu) / X_sigma
    pred_n = model.predict(noisy_val_n, verbose=0)
    pred = pred_n * Y_sigma + Y_mu
    err = np.abs(pred - Y_val)
    noise_rows.append({'noise_std': noise, 'mae_a': float(err[:, 0].mean()), 'mae_b': float(err[:, 1].mean()), 'mae_theta_deg': float(np.degrees(err[:, 2].mean()))})
pd.DataFrame(noise_rows)

### Note

When the noise level rises, the angle error often grows first. That matches the intuition that orientation is more sensitive.

## Task 3 Helpers

In [ ]:
def ellipse_mask(a, b, theta, cx, cy, X, Y):
    x_shift = X - cx
    y_shift = Y - cy
    x_rot = x_shift * np.cos(theta) + y_shift * np.sin(theta)
    y_rot = -x_shift * np.sin(theta) + y_shift * np.cos(theta)
    return ((x_rot / a) ** 2 + (y_rot / b) ** 2) <= 1.0

def make_task3_dataset(num_samples=400, img_size=16, seed=42):
    rng = np.random.default_rng(seed)
    a = 0.2 + 0.4 * rng.random(num_samples)
    b = 0.1 + 0.3 * rng.random(num_samples)
    theta = np.pi * rng.random(num_samples)
    cx = -0.5 + rng.random(num_samples)
    cy = -0.5 + rng.random(num_samples)
    x = np.linspace(-1.0, 1.0, img_size)
    X, Y = np.meshgrid(x, x)
    X_in = np.column_stack([a, b, theta, cx, cy]).astype(np.float32)
    Y_out = np.zeros((num_samples, img_size * img_size), dtype=np.float32)
    for i in range(num_samples):
        Y_out[i, :] = ellipse_mask(a[i], b[i], theta[i], cx[i], cy[i], X, Y).astype(np.float32).ravel()
    return X_in, Y_out, (x, X, Y)

def build_task3_model(input_dim, output_dim, hidden_units, l2_strength=0.0):
    modules = []
    prev_dim = input_dim
    for units in hidden_units:
        modules.append(nn.Linear(prev_dim, units))
        modules.append(nn.ReLU())
        prev_dim = units
    modules.append(nn.Linear(prev_dim, output_dim))
    model = nn.Sequential(*modules)
    model.weight_decay = l2_strength
    return model

def train_task3_model(model, X_train, Y_train, X_val=None, Y_val=None, epochs=5, batch_size=32):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=getattr(model, 'weight_decay', 0.0))
    criterion = nn.BCEWithLogitsLoss()
    train_loader = DataLoader(
        TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(Y_train, dtype=torch.float32)),
        batch_size=batch_size,
        shuffle=True,
    )
    history = {'loss': [], 'val_loss': []}
    for _ in range(epochs):
        model.train()
        losses = []
        for batch_X, batch_Y in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(batch_X), batch_Y)
            loss.backward()
            optimizer.step()
            losses.append(float(loss.item()))
        history['loss'].append(float(np.mean(losses)))
        if X_val is not None and Y_val is not None:
            model.eval()
            with torch.no_grad():
                val_loss = criterion(
                    model(torch.tensor(X_val, dtype=torch.float32)),
                    torch.tensor(Y_val, dtype=torch.float32),
                ).item()
            history['val_loss'].append(float(val_loss))
    return history

def evaluate_task3_model(model, X_val, Y_val):
    criterion = nn.BCEWithLogitsLoss()
    model.eval()
    with torch.no_grad():
        logits = model(torch.tensor(X_val, dtype=torch.float32))
        return float(criterion(logits, torch.tensor(Y_val, dtype=torch.float32)).item())

def predict_task3_masks(model, X_input):
    model.eval()
    with torch.no_grad():
        logits = model(torch.tensor(X_input, dtype=torch.float32)).cpu().numpy()
    return 1.0 / (1.0 + np.exp(-logits))


## Task 3: Resolution and Layer Depth

We compare a small image grid and a larger one, and also compare a shallow model with a deeper one.

In [ ]:
X16, Y16, _ = make_task3_dataset(num_samples=300, img_size=16, seed=1)
X32, Y32, _ = make_task3_dataset(num_samples=300, img_size=32, seed=1)

def normalize_pair(X):
    mu = X.mean(axis=0)
    sigma = X.std(axis=0) + 1e-8
    return (X - mu) / sigma, mu, sigma

def simple_split(X, Y):
    return X[:240], X[240:], Y[:240], Y[240:]

X16_train, X16_val, Y16_train, Y16_val = simple_split(X16, Y16)
X32_train, X32_val, Y32_train, Y32_val = simple_split(X32, Y32)

X16_train_n, X16_val_n, X16_mu, X16_sigma = normalize_train_val(X16_train, X16_val)
X32_train_n, X32_val_n, X32_mu, X32_sigma = normalize_train_val(X32_train, X32_val)

m16_shallow = build_task3_model(5, 16 * 16, [64])
m16_deep = build_task3_model(5, 16 * 16, [128, 64, 32])
m32_shallow = build_task3_model(5, 32 * 32, [64])

train_task3_model(m16_shallow, X16_train_n, Y16_train, epochs=5)
train_task3_model(m16_deep, X16_train_n, Y16_train, epochs=5)
train_task3_model(m32_shallow, X32_train_n, Y32_train, epochs=5)

task3_rows = []
for name, model, Xv, Yv in [
    ('16x16 shallow', m16_shallow, X16_val_n, Y16_val),
    ('16x16 deep', m16_deep, X16_val_n, Y16_val),
    ('32x32 shallow', m32_shallow, X32_val_n, Y32_val),
]:
    loss = evaluate_task3_model(model, Xv, Yv)
    task3_rows.append({'model': name, 'params': count_params(model), 'val_loss': loss})
pd.DataFrame(task3_rows)


### Note

The larger resolution model has a much larger output layer, so it usually needs more capacity or more training to match the smaller grid.

## Task 3: Regularization

We compare a baseline model and an L2-regularized model on the 32 x 32 case.

In [ ]:
baseline = build_task3_model(5, 32 * 32, [128, 64], l2_strength=0.0)
regularized = build_task3_model(5, 32 * 32, [128, 64], l2_strength=0.01)
train_task3_model(baseline, X32_train_n, Y32_train, X32_val_n, Y32_val, epochs=5)
train_task3_model(regularized, X32_train_n, Y32_train, X32_val_n, Y32_val, epochs=5)

regularization_rows = [
    {'model': 'baseline', 'val_loss': evaluate_task3_model(baseline, X32_val_n, Y32_val), 'params': count_params(baseline)},
    {'model': 'l2=0.01', 'val_loss': evaluate_task3_model(regularized, X32_val_n, Y32_val), 'params': count_params(regularized)},
]
pd.DataFrame(regularization_rows)


## Notes to Keep

- Add layers when the model is clearly underfitting.
- Remove layers when the model is too slow or over-parameterized.
- Increase `N` only if the task really needs extra detail.
- Add noise during testing to check robustness.
- For Task 3, check hard ellipses: thin, rotated, and boundary-touching shapes.